# vigilex – Phase I
## Notebook 02: Feature Engineering

**Ziel:**
- MAUDE-Rohdaten aus Notebook 01 laden
- Rolling-Window Complaint-Features bauen (7, 30, 90 Tage)
- Severity Score aus `event_type` und `patient_outcome` ableiten
- Recall-Labels joinen (MAUDE × FDA Recall DB)
- Feature-Matrix als Parquet speichern für Notebook 03

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv('OPENFDA_API_KEY', '')

DATA_RAW  = Path('../data/raw')
DATA_PROC = Path('../data/processed')
DATA_PROC.mkdir(parents=True, exist_ok=True)

print('Setup OK')

## 1. Rohdaten laden

In [ ]:
df = pd.read_parquet(DATA_RAW / 'maude_insulinpumpen.parquet')

# Datumsspalten sicherstellen
for col in ['date_received', 'date_of_event']:
    if df[col].dtype == object:
        df[col] = pd.to_datetime(df[col], format='%Y%m%d', errors='coerce')
    else:
        df[col] = pd.to_datetime(df[col], errors='coerce')

print(f'Shape: {df.shape}')
print(f'Zeitraum: {df["date_of_event"].min().date()} – {df["date_of_event"].max().date()}')
df.head(3)

## 2. Severity Score

Ordinalskala basierend auf `event_type` und `patient_outcome`:
- `event_type`: Death (4) > Injury (3) > Malfunction (2) > Other (1)
- `patient_outcome`: Death (5) > Life Threatening (4) > Hospitalization (3) > Required Intervention (2) > No Consequences (1)

In [ ]:
EVENT_TYPE_SCORE = {
    'D':  4,   # Death
    'IN': 3,   # Injury
    'IL': 3,   # Injury (Life Threatening)
    'M':  2,   # Malfunction
    'O':  1,   # Other
    'N':  1,   # No answer provided
}

OUTCOME_SCORE = {
    'D':  5,   # Death
    'LT': 4,   # Life Threatening
    'H':  3,   # Hospitalization
    'RI': 2,   # Required Intervention
    'O':  1,   # Other
    'N':  1,   # No consequences
    '*':  1,   # Unknown
}

df['event_score']   = df['event_type'].map(EVENT_TYPE_SCORE).fillna(1).astype(int)
df['outcome_score'] = df['patient_outcome'].map(OUTCOME_SCORE).fillna(1).astype(int)
df['severity_score'] = df[['event_score', 'outcome_score']].max(axis=1)

print('Severity Score Verteilung:')
print(df['severity_score'].value_counts().sort_index())

## 3. Rolling-Window Features pro Hersteller

Für jeden Report berechnen wir rückblickend:
- Anzahl Complaints (7, 30, 90 Tage)
- Anzahl schwerer Events (severity ≥ 3) in denselben Fenstern
- Complaint-Beschleunigung (30d / 90d Ratio)

Granularität: **Hersteller × Produkt-Code × Datum**

In [ ]:
def build_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    """Berechnet Rolling-Window Features pro Hersteller + Produktcode."""
    df = df.copy()
    df = df.dropna(subset=['date_of_event', 'manufacturer_name'])
    df = df.sort_values('date_of_event').reset_index(drop=True)

    result_parts = []
    groups = df.groupby(['manufacturer_name', 'product_code'], sort=False)

    for (mfr, pc), grp in groups:
        grp = grp.set_index('date_of_event').sort_index()

        # Tages-Aggregation
        daily = grp.resample('D').agg(
            complaints=('report_number', 'count'),
            severe_events=('severity_score', lambda x: (x >= 3).sum()),
            max_severity=('severity_score', 'max'),
            mean_severity=('severity_score', 'mean'),
        )

        for window, label in [(7, '7d'), (30, '30d'), (90, '90d')]:
            rolled = daily['complaints'].rolling(f'{window}D', min_periods=1).sum()
            daily[f'complaints_{label}']     = rolled
            severe = daily['severe_events'].rolling(f'{window}D', min_periods=1).sum()
            daily[f'severe_events_{label}'] = severe

        # Beschwerde-Beschleunigung: 30d vs 90d
        daily['complaint_accel'] = (
            daily['complaints_30d'] / (daily['complaints_90d'] + 1e-9)
        )

        daily['manufacturer_name'] = mfr
        daily['product_code']      = pc
        result_parts.append(daily.reset_index())

    return pd.concat(result_parts, ignore_index=True)


print('Berechne Rolling Features...')
df_features = build_rolling_features(df)
print(f'Feature-DataFrame Shape: {df_features.shape}')
df_features.head(3)

## 4. Recall-Labels laden und joinen

**Logik:** Ein Hersteller-Produkt bekommt Label `recall=1` wenn innerhalb von **12 Monaten nach einem Datenpunkt** ein Recall für dieses Gerät erscheint.

In [ ]:
RECALL_URL = 'https://api.fda.gov/device/recall.json'

def fetch_all_recalls(search: str, api_key: str = '', max_records: int = 5000) -> pd.DataFrame:
    """Lädt alle passenden Recalls aus der FDA Recall DB."""
    all_recs = []
    skip = 0

    while skip < max_records:
        params = {'search': search, 'limit': 100, 'skip': skip}
        if api_key:
            params['api_key'] = api_key

        r = requests.get(RECALL_URL, params=params, timeout=30)
        if r.status_code != 200:
            print(f'Fehler bei skip={skip}: {r.status_code}')
            break

        batch = r.json().get('results', [])
        if not batch:
            break

        all_recs.extend(batch)
        skip += len(batch)
        print(f'{skip} Recalls geladen...', end='\r')
        time.sleep(0.15)

    print(f'\nGesamt: {len(all_recs)} Recalls')
    return pd.DataFrame(all_recs)


df_recalls = fetch_all_recalls(
    search='product_description:"insulin pump"',
    api_key=API_KEY
)
df_recalls.head(3)

In [ ]:
# Recall-Datum parsen + relevante Spalten behalten
df_recalls['recall_date'] = pd.to_datetime(
    df_recalls['recall_initiation_date'], errors='coerce'
)

recalls_clean = df_recalls[['recall_date', 'recalling_firm', 'classification', 'product_code']].dropna(
    subset=['recall_date', 'recalling_firm']
).copy()

# Firmenname normalisieren (Großbuchstaben)
recalls_clean['recalling_firm_norm'] = recalls_clean['recalling_firm'].str.upper().str.strip()

print(f'Recalls nach Klasse:')
print(recalls_clean['classification'].value_counts())

In [ ]:
def label_recall_risk(df_feat: pd.DataFrame, recalls: pd.DataFrame, horizon_days: int = 365) -> pd.DataFrame:
    """
    Fügt `recall_label` und `recall_class` hinzu.
    recall_label = 1 wenn innerhalb `horizon_days` ab dem Feature-Datum
    ein Recall für denselben Hersteller existiert.
    """
    df_feat = df_feat.copy()
    df_feat['recall_label'] = 0
    df_feat['recall_class'] = 'None'

    mfr_norm = df_feat['manufacturer_name'].str.upper().str.strip()
    df_feat['_mfr_norm'] = mfr_norm

    for idx, row in df_feat.iterrows():
        horizon_end = row['date_of_event'] + pd.Timedelta(days=horizon_days)
        matches = recalls[
            (recalls['recalling_firm_norm'].str.contains(
                row['_mfr_norm'][:10], na=False, regex=False)
            ) &
            (recalls['recall_date'] >= row['date_of_event']) &
            (recalls['recall_date'] <= horizon_end)
        ]
        if len(matches) > 0:
            df_feat.at[idx, 'recall_label'] = 1
            df_feat.at[idx, 'recall_class'] = matches['classification'].iloc[0]

    df_feat.drop(columns=['_mfr_norm'], inplace=True)
    return df_feat


print('Labele Recall-Risiko (kann etwas dauern)...')
df_labeled = label_recall_risk(df_features, recalls_clean)

print('\nRecall-Label Verteilung:')
print(df_labeled['recall_label'].value_counts())
print(f'Recall Rate: {df_labeled["recall_label"].mean():.1%}')

## 5. Feature-Selektion & Speichern

In [ ]:
FEATURE_COLS = [
    'complaints_7d', 'complaints_30d', 'complaints_90d',
    'severe_events_7d', 'severe_events_30d', 'severe_events_90d',
    'complaint_accel',
    'max_severity', 'mean_severity',
]
TARGET_COL = 'recall_label'
META_COLS  = ['date_of_event', 'manufacturer_name', 'product_code', 'recall_class']

df_ml = df_labeled[META_COLS + FEATURE_COLS + [TARGET_COL]].dropna(subset=FEATURE_COLS)

print(f'ML-Dataset Shape: {df_ml.shape}')
print(f'Features: {FEATURE_COLS}')
print(f'Class Balance: {df_ml[TARGET_COL].value_counts(normalize=True).round(3).to_dict()}')

df_ml.to_parquet(DATA_PROC / 'features_labeled.parquet', index=False)
print(f'\nGespeichert: {DATA_PROC / "features_labeled.parquet"}')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ['complaints_30d', 'severe_events_30d', 'complaint_accel']):
    sns.boxplot(
        data=df_ml, x='recall_label', y=col, ax=ax,
        palette={0: 'steelblue', 1: 'coral'}
    )
    ax.set_title(col)
    ax.set_xlabel('Recall (0=Nein, 1=Ja)')

plt.suptitle('Feature-Verteilung nach Recall-Label', y=1.02)
plt.tight_layout()
plt.savefig('../data/feature_distributions.png', dpi=150)
plt.show()

## Zusammenfassung

| Feature | Beschreibung |
|---|---|
| `complaints_7/30/90d` | Anzahl Complaints im Rolling Window |
| `severe_events_7/30/90d` | Events mit severity ≥ 3 im Window |
| `complaint_accel` | 30d/90d Ratio – misst Beschwerde-Anstieg |
| `max_severity` | Schwerster Event im Aggregations-Tag |
| `mean_severity` | Durchschnittlicher Schweregrad |
| `recall_label` | **Target**: Recall innerhalb 12 Monate |

→ Weiter mit **Notebook 03: Modellierung**